# Interactive Vermont Geology Explorer
## A Geospatial Python Workshop

---

### Workshop Overview

Welcome! This notebook demonstrates a complete geospatial workflow using real-world Vermont GIS data. You'll learn to:

- **Fetch data from REST APIs**: Query ArcGIS REST endpoints to retrieve GeoJSON data
- **Build interactive visualizations**: Create dynamic maps with user controls
- **Perform spatial operations**: Clip, intersect, and analyze vector geometries
- **Analyze spatial data**: Calculate statistics and create visualizations
- **Optimize workflows**: Implement caching to reduce redundant API calls

### What We're Building

By the end of this notebook, you'll have an interactive tool that allows you to:
1. Select any Vermont town from a dropdown menu
2. View bedrock geology on an interactive map
3. Extract geology data for that specific town
4. Analyze the distribution of geologic units
5. Visualize results with charts and tables

### Learning Objectives

**Data Access**
- Query ArcGIS REST Feature Services and Map Services
- Understand REST API parameter structures
- Implement local caching strategies

**Geospatial Analysis**
- Work with coordinate reference systems (CRS)
- Perform spatial queries (bounding box intersections)
- Clip geometries using spatial overlays
- Calculate areas for polygon features

**Interactive Visualization**
- Create interactive web maps with Python
- Add user interface controls (dropdowns, buttons)
- Implement hover and click interactions
- Layer different data sources effectively

**Python Geospatial Ecosystem**
- Use GeoPandas for vector data manipulation
- Apply Shapely for geometric operations
- Build maps with Folium or ipyleaflet
- Leverage ipywidgets for interactivity

### Data Sources

This workshop uses public data from Vermont state agencies:

**1. Town Boundaries**
- **Source**: Vermont Center for Geographic Information (VCGI)
- **Endpoint**: VCGI OpenData Boundary Service
- **License**: Public domain
- **Format**: GeoJSON via ArcGIS REST API

**2. Bedrock Geology**
- **Source**: Vermont Agency of Natural Resources (ANR)
- **Endpoint**: ANR Geologic Map Service
- **License**: Public domain
- **Format**: GeoJSON via query (vector) and tile service (raster)

**3. Basemap**
- **Source**: National Geographic / Esri
- **Type**: Vector tile layer
- **Purpose**: Provides geographic context

### Prerequisites

**Python Knowledge**
- Basic Python syntax and data structures
- Familiarity with pandas DataFrames helpful but not required

**GIS Concepts**
- Basic understanding of coordinate systems (will be explained)
- Familiarity with vector data (points, lines, polygons)
- No prior GIS software experience required!

### Workflow Overview

```
1. Setup & Imports
   ↓
2. Fetch Town Boundaries → Cache Locally
   ↓
3. Create Town Selector UI
   ↓
4. Display Map with Geology Layer
   ↓
5. Query Geology for Selected Town → Cache Locally
   ↓
6. Clip Geology to Town Boundary
   ↓
7. Add Clipped Layer to Map (Interactive)
   ↓
8. Analyze & Visualize Results
```

---

### About This Notebook

This educational material was developed openly and transparently with AI assistance from Claude Code. The workflow demonstrates real-world geospatial analysis patterns you can adapt for your own projects.

**Repository**: [PyDataVT2025](https://github.com/yourusername/PyDataVT2025)

---

Let's get started!

## Environment Setup

First, let's import the required libraries. If you don't have these installed, see the `requirements.txt` file in the repository.

### Required Libraries:

- **`geopandas`**: GeoPandas extends pandas to work with geospatial data. It combines the capabilities of pandas with geometric operations from shapely.
- **`shapely`**: Library for geometric operations (automatically installed with geopandas)
- **`folium`**: Creates interactive Leaflet maps in Python
- **`requests`**: Simple HTTP library for making API calls
- **`ipywidgets`**: Interactive UI components for Jupyter notebooks
- **`matplotlib`**: Plotting library for charts and visualizations
- **`pathlib`**: Object-oriented filesystem paths (part of standard library)

### Coordinate Reference Systems (CRS)

We'll be working with two coordinate systems:
- **EPSG:32145** - Vermont State Plane (NAD83) - Used by Vermont state data, units in meters
- **EPSG:4326** - WGS84 - Standard for web maps (Google Maps, OpenStreetMap), units in degrees

In [ ]:
# Standard library imports
import json
from pathlib import Path
from typing import Optional, Tuple
import warnings

# Third-party imports
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box
import folium
from folium import plugins
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Button, Output, VBox, HBox
from IPython.display import display, HTML

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"\nLibrary Versions:")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Folium: {folium.__version__}")

### Create Data Directory

We'll cache downloaded data locally to avoid repeated API calls. This is a best practice when working with external data sources:
- Faster execution on subsequent runs
- Reduces load on data providers
- Enables offline work after initial download
- Makes your analysis reproducible

In [ ]:
# Create data directory if it doesn't exist
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"✓ Data directory ready: {DATA_DIR.absolute()}")

### Configuration

Let's define the URLs and parameters we'll use throughout the notebook. Keeping these at the top makes the code easier to maintain and adapt for other regions or data sources.

In [ ]:
# API Endpoints
TOWN_BOUNDARIES_URL = (
    "https://services1.arcgis.com/BkFxaEFNwHqX3tAw/arcgis/rest/services/"
    "FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1/FeatureServer/0/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

GEOLOGY_MAPSERVICE_URL = (
    "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/"
    "OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/165"
)

GEOLOGY_QUERY_ENDPOINT = f"{GEOLOGY_MAPSERVICE_URL}/query"

BASEMAP_URL = (
    "https://basemaps.arcgis.com/arcgis/rest/services/"
    "World_Basemap_v2/VectorTileServer"
)

# Coordinate Reference Systems
VT_STATE_PLANE = "EPSG:32145"  # Vermont State Plane NAD83 (meters)
WEB_MERCATOR = "EPSG:3857"      # Web Mercator (for tile services)
WGS84 = "EPSG:4326"             # WGS84 (latitude/longitude)

# File paths for cached data
TOWNS_CACHE = DATA_DIR / "towns.geojson"

print("✓ Configuration complete!")
print(f"\nData Sources:")
print(f"  Towns: VCGI OpenData Portal")
print(f"  Geology: VT Agency of Natural Resources")
print(f"\nCoordinate Systems:")
print(f"  Vermont State Plane: {VT_STATE_PLANE}")
print(f"  WGS84 (Web): {WGS84}")

---

## Ready to Begin!

With our environment configured, we're ready to start fetching and working with geospatial data. In the next section, we'll download Vermont town boundaries and explore the data structure.

### Key Concepts to Remember:

1. **Caching**: We save downloaded data locally to avoid repeated API calls
2. **CRS**: Always be aware of which coordinate system your data uses
3. **REST APIs**: We'll interact with ArcGIS REST services using simple HTTP requests
4. **GeoJSON**: A standard format for encoding geographic data structures

Let's dive in! 🗺️